In [1]:
import os
import numpy as np
import pandas as pd

# Paths: notebook is in zavala_electricity_market, data in dataset/IM-3-GO-WEST
# Run from project root (prj_market) or from zavala_electricity_market
_cwd = os.getcwd()
if os.path.basename(_cwd) == "zavala_electricity_market":
    BASE_DIR = os.path.dirname(_cwd)
else:
    BASE_DIR = _cwd
DATA_DIR = os.path.join(BASE_DIR, "dataset", "IM-3-GO-WEST")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(_cwd, "dataset", "IM-3-GO-WEST")

# Ensure zavala_electricity_market is on path (when running from project root)
import sys
ZAVALA_DIR = os.path.join(BASE_DIR, "zavala_electricity_market")
if os.path.isdir(ZAVALA_DIR) and ZAVALA_DIR not in sys.path:
    sys.path.insert(0, ZAVALA_DIR)

# --- Minimal customization for current repo layout ---

# 1) If the original DATA_DIR doesn't exist, try archive/dataset/IM-3-GO-WEST
if not os.path.isdir(DATA_DIR):
    alt_data_dir = os.path.join(BASE_DIR, "archive", "dataset", "IM-3-GO-WEST")
    if os.path.isdir(alt_data_dir):
        DATA_DIR = alt_data_dir

# 2) If ZAVALA_DIR isn't a valid dir (e.g. code is at repo root),
#    fall back to BASE_DIR itself when it has zavala_funcs.py
if not os.path.isdir(ZAVALA_DIR):
    repo_root_candidate = BASE_DIR
    if os.path.isfile(os.path.join(repo_root_candidate, "zavala_funcs.py")):
        if repo_root_candidate not in sys.path:
            sys.path.insert(0, repo_root_candidate)
        ZAVALA_DIR = repo_root_candidate

print("Using DATA_DIR:", DATA_DIR)
print("Using ZAVALA_DIR:", ZAVALA_DIR)
print("Files:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "not found")

from zavala_funcs import (
    zavala,
    zavala_cvar,
    zavala_deterministic_da,
    zavala_rt_energy_only,
    expected_caps_from_scenarios,
    price_distortion,
    probability_feasible,
    expected_cumulative_regret,
    compute_social_surplus,
    tail_worst_indices_by_value,
    _stack_rt,
)

print("Data dir:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "not found")

Using DATA_DIR: /Users/maxwirattawut/Developer/zavala_electricity_market/dataset/IM-3-GO-WEST
Using ZAVALA_DIR: /Users/maxwirattawut/Developer/zavala_electricity_market
Files: ['nodal_load.csv', 'nodal_wind.csv', 'thermal_gens.csv', 'egrid2023_data_rev2.xlsx', 'nodal_solar.csv']
Data dir: /Users/maxwirattawut/Developer/zavala_electricity_market/dataset/IM-3-GO-WEST
Files: ['nodal_load.csv', 'nodal_wind.csv', 'thermal_gens.csv', 'egrid2023_data_rev2.xlsx', 'nodal_solar.csv']


In [2]:
# Load nodal time series (rows = time, columns = bus_XXXXX)
solar = pd.read_csv(os.path.join(DATA_DIR, "nodal_solar.csv"))
wind = pd.read_csv(os.path.join(DATA_DIR, "nodal_wind.csv"))
load_df = pd.read_csv(os.path.join(DATA_DIR, "nodal_load.csv"))
thermal_df = pd.read_csv(os.path.join(DATA_DIR, "thermal_gens.csv"))

T = len(solar)
assert len(wind) == T and len(load_df) == T, "Solar, wind, load must have same length"
print(f"Time steps: {T}")
print(f"Solar columns: {solar.shape[1]}, Wind: {wind.shape[1]}, Load: {load_df.shape[1]}")
print(f"Thermal generators: {len(thermal_df)}")

Time steps: 8760
Solar columns: 125, Wind: 125, Load: 125
Thermal generators: 280


In [3]:
# Choose buses with non-trivial solar: columns with max > threshold
solar_cols = [c for c in solar.columns if solar[c].max() > 50]
wind_cols = [c for c in wind.columns if wind[c].max() > 50]
# Pick x solar and y wind (unreliable)
num_solar, num_wind = 9, 9
num_thermal = 9
solar_buses = solar_cols[:num_solar] if len(solar_cols) >= num_solar else list(solar.columns[:num_solar])
wind_buses = wind_cols[:num_wind] if len(wind_cols) >= num_wind else list(wind.columns[:num_wind])

# Reliable: aggregate thermal by bus, pick 4 buses with largest capacity
thermal_by_bus = thermal_df.groupby("Bus")["Max_Cap"].sum().sort_values(ascending=False)
thermal_buses_numeric = list(thermal_by_bus.head(num_thermal).index)  # e.g. [408441, 135041, ...]
thermal_bus_cols = [f"bus_{b}" for b in thermal_buses_numeric]  # for load alignment if needed

# Load: use total system load (sum over all buses)
load_total = load_df.sum(axis=1).values  # (T,)

print("Solar buses (unreliable):", solar_buses)
print("Wind buses (unreliable):", wind_buses)
print("Thermal buses (reliable):", thermal_buses_numeric)
print("Load: system total (sum over all buses)")

Solar buses (unreliable): ['bus_100931', 'bus_200261', 'bus_201361', 'bus_204901', 'bus_211211', 'bus_230601', 'bus_232301', 'bus_232631', 'bus_251601']
Wind buses (unreliable): ['bus_100931', 'bus_102281', 'bus_105701', 'bus_106771', 'bus_134461', 'bus_135041', 'bus_211211', 'bus_211651', 'bus_251601']
Thermal buses (reliable): [500991, 605141, 408441, 261001, 403691, 261331, 606631, 280991, 102281]
Load: system total (sum over all buses)


In [4]:
# Filter thermal_df for the selected buses
selected_buses = [500991, 605141, 408441, 261001]
thermal_selected = thermal_df[thermal_df['Bus'].isin(selected_buses)]
display(thermal_selected)

,Name,Bus,Fuel,Max_Cap,Min_Cap,Heat_Rate
0,PHOENIX_Nuc,408441,NUC (Nuclear),4209.60,1046.90,8.530000
1,TONOPAH_NG,408441,NG (Natural Gas),1325.10,393.07,4.177264
2,PHOENIX_NG,408441,NG (Natural Gas),1207.38,449.72,3.622784
28,BRUSH_C,605141,BIT (Bituminous Coal),552.30,78.79,8.941106
29,WELLINGTON_C,605141,BIT (Bituminous Coal),800.40,247.36,8.242654
30,DENVER_C,605141,BIT (Bituminous Coal),586.31,164.07,7.933221
31,KEENESBURG_NG,605141,NG (Natural Gas),685.11,246.77,5.015047
32,BOULDER_C,605141,BIT (Bituminous Coal),251.00,55.58,8.227582
33,PLATTEVILLE_NG,605141,NG (Natural Gas),1185.48,481.17,2.758167
34,AURORA_NG,605141,NG (Natural Gas),397.80,147.60,4.898308


Debug Logs

In [5]:
# For diagnostics: breakdown of DA and RT allocations
def _log(msg, logfile=None):
    if logfile is None:
        print(msg)
    else:
        with open(logfile, "a", encoding="utf-8") as f:
            f.write(str(msg) + "\n")

def _tech_breakdown_da(g_da):
    g_da = np.asarray(g_da, dtype=float)
    return {
        "solar": g_da[:3].sum(),
        "wind": g_da[3:6].sum(),
        "thermal": g_da[6:10].sum(),
        "total": g_da.sum(),
    }

def _tech_breakdown_rt(G_rt, probs=None):
    G_rt = np.asarray(G_rt, dtype=float)  # shape (S, 10)
    solar = G_rt[:, :3].sum(axis=1)
    wind = G_rt[:, 3:6].sum(axis=1)
    thermal = G_rt[:, 6:10].sum(axis=1)
    total = G_rt.sum(axis=1)

    if probs is None:
        return {
            "solar": solar.mean(),
            "wind": wind.mean(),
            "thermal": thermal.mean(),
            "total": total.mean(),
        }

    probs = np.asarray(probs, dtype=float)
    return {
        "solar": np.dot(probs, solar),
        "wind": np.dot(probs, wind),
        "thermal": np.dot(probs, thermal),
        "total": np.dot(probs, total),
    }

def _print_case_diag(name, probs, g_da, d_da, G_rt, D_rt, pi, Pi,
                     n_solar, n_wind, n_thermal, logfile=None):
    g_da = np.asarray(g_da, dtype=float)
    d_da = np.asarray(d_da, dtype=float)
    G_rt = np.asarray(G_rt, dtype=float)
    D_rt = np.asarray(D_rt, dtype=float)
    probs = np.asarray(probs, dtype=float)
    Pi = np.asarray(Pi, dtype=float)

    exp_rt = np.dot(probs, G_rt)

    solar_slice = slice(0, n_solar)
    wind_slice = slice(n_solar, n_solar + n_wind)
    thermal_slice = slice(n_solar + n_wind, n_solar + n_wind + n_thermal)

    da_alloc = {
        "solar": g_da[solar_slice].sum(),
        "wind": g_da[wind_slice].sum(),
        "thermal": g_da[thermal_slice].sum(),
        "total": g_da.sum(),
    }
    rt_alloc = {
        "solar": exp_rt[solar_slice].sum(),
        "wind": exp_rt[wind_slice].sum(),
        "thermal": exp_rt[thermal_slice].sum(),
        "total": exp_rt.sum(),
    }

    da_load = float(d_da.sum())
    exp_rt_load = float(np.dot(probs, D_rt.sum(axis=1)))

    _log(f"\n===== {name} =====", logfile)
    _log(f"DA price pi: {float(pi):.6f}", logfile)
    _log(f"E[RT price]: {float(np.dot(probs, Pi)):.6f}", logfile)
    _log(f"DA load: {da_load:.6f}", logfile)
    _log(f"E[RT load]: {exp_rt_load:.6f}", logfile)
    _log("DA allocation by tech: " + str({k: round(v, 4) for k, v in da_alloc.items()}), logfile)
    _log("Expected RT allocation by tech: " + str({k: round(v, 4) for k, v in rt_alloc.items()}), logfile)

    gen_names = (
        [f"solar_{i+1}" for i in range(n_solar)] +
        [f"wind_{i+1}" for i in range(n_wind)] +
        [f"thermal_{i+1}" for i in range(n_thermal)]
    )

    df = pd.DataFrame({
        "gen": gen_names,
        "DA": g_da,
        "E_RT": exp_rt,
        "RT_minus_DA": exp_rt - g_da,
    })
    _log(df.to_string(index=False), logfile)
    
def _print_stoch_vs_cvar_diff(z_g_i, cvar_g_i, n_solar, n_wind, n_thermal, logfile=None):
    z = np.asarray(z_g_i, dtype=float)
    c = np.asarray(cvar_g_i, dtype=float)

    solar_slice = slice(0, n_solar)
    wind_slice = slice(n_solar, n_solar + n_wind)
    thermal_slice = slice(n_solar + n_wind, n_solar + n_wind + n_thermal)

    gen_names = (
        [f"solar_{i+1}" for i in range(n_solar)] +
        [f"wind_{i+1}" for i in range(n_wind)] +
        [f"thermal_{i+1}" for i in range(n_thermal)]
    )

    df = pd.DataFrame({
        "gen": gen_names,
        "stoch_DA": z,
        "cvar_DA": c,
        "cvar_minus_stoch": c - z,
    })

    _log("\n===== CVaR - Stochastic DA difference =====", logfile)
    _log(df.to_string(index=False), logfile)
    _log("Tech-level difference: " + str({
        "solar": round((c[solar_slice] - z[solar_slice]).sum(), 4),
        "wind": round((c[wind_slice] - z[wind_slice]).sum(), 4),
        "thermal": round((c[thermal_slice] - z[thermal_slice]).sum(), 4),
        "total": round((c - z).sum(), 4),
    }), logfile)

In [6]:
# Initialize log files
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

debug_log = os.path.join(log_dir, "zavala_debug_log.txt")
with open(debug_log, "w", encoding="utf-8") as f:
    f.write("Zavala diagnostics log\n")

Build Real Data

In [7]:
def build_real_data_instance(solar_df, wind_df, load_total_vec, thermal_by_bus, thermal_buses_numeric,
                              solar_buses, wind_buses, start_idx, num_scenarios, rng=None):
    """
    Build (probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar) from real data for one time window.
    - start_idx: first time index
    - num_scenarios: number of consecutive time steps (scenarios)
    """
    if rng is None:
        rng = np.random.default_rng()
    end_idx = start_idx + num_scenarios
    S = num_scenarios

    # Scenario probabilities: near-uniform (same idea as s_real10_mix)
    kappa = 1500.0
    alpha = np.full(S, kappa / S)
    probs = rng.dirichlet(alpha)
    probs = probs / probs.sum()

    # Marginal costs: cheap for unreliable (solar/wind), higher for reliable (thermal)
    n_solar = len(solar_buses)
    n_wind = len(wind_buses)
    n_unrel = n_solar + n_wind
    n_rel = len(thermal_buses_numeric)

    mc_unrel = rng.uniform(8.0, 14.0, size=n_unrel)
    mc_rel = rng.uniform(35.0, 55.0, size=n_rel)
    mc_g_i = np.concatenate([mc_unrel, mc_rel]).astype(float)

    # Single inelastic load (VOLL)
    mv_d_j = np.array([1000.0], dtype=float)

    # Generator capacities per scenario (S x 10)
    # Columns 0..2: solar, 3..5: wind, 6..9: thermal (constant)
    solar_vals = solar_df.loc[start_idx:end_idx - 1, solar_buses].values  # (S, 3)
    wind_vals = wind_df.loc[start_idx:end_idx - 1, wind_buses].values    # (S, 3)
    unrel_caps = np.clip(np.hstack([solar_vals, wind_vals]), 0.0, None)  # (S, 6)

    rel_caps = np.array([thermal_by_bus[b] for b in thermal_buses_numeric], dtype=float)
    rel_caps = np.broadcast_to(rel_caps, (S, len(thermal_buses_numeric)))  # (S, 4) constant across scenarios

    g_i_bar = np.hstack([unrel_caps, rel_caps])  # (S, 10)

    # Demand: system total load for each scenario (S x 1)
    d_j_bar = load_total_vec[start_idx:end_idx].reshape(-1, 1).astype(float)
    d_j_bar = np.clip(d_j_bar, 1e-6, None)  # avoid zeros

    return probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar

In [8]:
# Quick sanity check: one small window
NUM_SCENARIOS = 500
rng = np.random.default_rng(42)
probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
    solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
    solar_buses, wind_buses, start_idx=0, num_scenarios=min(NUM_SCENARIOS, T), rng=rng
)
print("probs.shape:", probs.shape, "sum:", probs.sum())
print("g_i_bar.shape:", g_i_bar.shape)
print("d_j_bar.shape:", d_j_bar.shape)
print("Unreliable (solar+wind) sample mean:", g_i_bar[:, :6].mean(axis=0))
print("Reliable (thermal) constant:", g_i_bar[0, 6:])
print("Load sample:", d_j_bar[:5].ravel())

probs.shape: (500,) sum: 1.0
g_i_bar.shape: (500, 27)
d_j_bar.shape: (500, 1)
Unreliable (solar+wind) sample mean: [ 67.33335058  17.28        28.36311076 390.34965713 131.82143061
  24.22398934]
Reliable (thermal) constant: [0.00000000e+00 0.00000000e+00 0.00000000e+00 1.26796677e+03
 2.58000000e+02 4.17920476e+02 2.42095982e+01 2.93740174e+02
 4.60995965e+02 6.63495200e+01 1.68442245e+01 6.00000000e-01
 6.92233000e+03 6.83762000e+03 6.74208000e+03 6.52169000e+03
 4.90857000e+03 4.80050000e+03 4.64808000e+03 4.38565000e+03
 4.24913000e+03]
Load sample: [75073. 72985. 71226. 70105. 69772.]


In [9]:
def run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar, logfile=None):
    """Run stochastic, CVaR, and deterministic Zavala for one instance. Returns dict of metrics.
    Same logic as run_zavala.py, no changes to external files.
    """
    # ----- Stochastic Zavala -----
    z_g_i, z_d_j, Z_G, Z_D, z_pi, z_Pi = zavala(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    prob_f = probability_feasible(probs, z_g_i, z_d_j, g_i_bar, d_j_bar)
    z_dist = price_distortion(probs, z_pi, z_Pi)
    z_reg = expected_cumulative_regret(probs, z_g_i, z_d_j, z_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_stoch = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=z_g_i, d_da=z_d_j, G_rt=Z_G, D_rt=Z_D)

    # # ----- CVaR Zavala -----
    cvar_g_i, cvar_d_j, C_G, C_D, cvar_pi, cvar_Pi, _ = zavala_cvar(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    cvar_dist = price_distortion(probs, cvar_pi, cvar_Pi)
    cvar_reg = expected_cumulative_regret(probs, cvar_g_i, cvar_d_j, cvar_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_cvar = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=cvar_g_i, d_da=cvar_d_j, G_rt=C_G, D_rt=C_D)

    # ----- Deterministic (expected capacities) -----
    gbar_det, dbar_det = expected_caps_from_scenarios(probs, g_i_bar, d_j_bar)
    g_det, d_det, pi_det = zavala_deterministic_da(mc_g_i, mv_d_j, gbar_det, dbar_det)
    G_det_list, D_det_list, Pi_det_list = [], [], []
    for p in range(len(probs)):
        Gp, Dp, Pi_p = zavala_rt_energy_only(mc_g_i, mv_d_j, g_det, d_det, g_i_bar[p], d_j_bar[p])
        G_det_list.append(Gp)
        D_det_list.append(Dp)
        Pi_det_list.append(Pi_p)
    G_det_rt, D_det_rt = _stack_rt(G_det_list, D_det_list)
    Pi_det = np.array(Pi_det_list)
    det_dist = price_distortion(probs, pi_det, Pi_det)
    det_reg = expected_cumulative_regret(probs, g_det, d_det, pi_det, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_det = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=g_det, d_da=d_det, G_rt=G_det_rt, D_rt=D_det_rt)

    # ----- Tail metrics (5% worst by high neg-surplus) -----
    tail = 0.05
    stoch_tail_idx = tail_worst_indices_by_value(ss_stoch["ss_per_scenario"], probs, tail=tail, worst="high")
    cvar_tail_idx = tail_worst_indices_by_value(ss_cvar["ss_per_scenario"], probs, tail=tail, worst="high")
    det_tail_idx = tail_worst_indices_by_value(ss_det["ss_per_scenario"], probs, tail=tail, worst="high")

    stoch_tail_welfare = -np.mean(ss_stoch["ss_per_scenario"][stoch_tail_idx])
    cvar_tail_welfare = -np.mean(ss_cvar["ss_per_scenario"][cvar_tail_idx])
    det_tail_welfare = -np.mean(ss_det["ss_per_scenario"][det_tail_idx])

    stoch_tail_dist = np.mean(np.abs(z_pi - np.array(z_Pi)[stoch_tail_idx]))
    cvar_tail_dist = np.mean(np.abs(cvar_pi - np.array(cvar_Pi)[cvar_tail_idx]))
    det_tail_dist = np.mean(np.abs(pi_det - Pi_det[det_tail_idx]))

    # Log diagnostics/output results analysis
    n_solar = len(solar_buses)
    n_wind = len(wind_buses)
    n_thermal = len(thermal_buses_numeric)

    _print_case_diag("Stochastic", probs, z_g_i, z_d_j, Z_G, Z_D, z_pi, z_Pi,
                    n_solar=n_solar, n_wind=n_wind, n_thermal=n_thermal, logfile=logfile)
    _print_case_diag("CVaR", probs, cvar_g_i, cvar_d_j, C_G, C_D, cvar_pi, cvar_Pi,
                    n_solar=n_solar, n_wind=n_wind, n_thermal=n_thermal, logfile=logfile)
    _print_case_diag("Deterministic", probs, g_det, d_det, G_det_rt, D_det_rt, pi_det, Pi_det,
                    n_solar=n_solar, n_wind=n_wind, n_thermal=n_thermal, logfile=logfile)
    _print_stoch_vs_cvar_diff(z_g_i, cvar_g_i,
                            n_solar=n_solar, n_wind=n_wind, n_thermal=n_thermal, logfile=logfile)

    return {
        "prob_feasible": prob_f,
        "stoch_distortion": z_dist, "stoch_regret": z_reg, "stoch_ss": ss_stoch["E_social_surplus"],
        "stoch_tail_welfare": stoch_tail_welfare, "stoch_tail_distortion": stoch_tail_dist,
        "cvar_distortion": cvar_dist, "cvar_regret": cvar_reg, "cvar_ss": ss_cvar["E_social_surplus"],
        "cvar_tail_welfare": cvar_tail_welfare, "cvar_tail_distortion": cvar_tail_dist,
        "det_distortion": det_dist, "det_regret": det_reg, "det_ss": ss_det["E_social_surplus"],
        "det_tail_welfare": det_tail_welfare, "det_tail_distortion": det_tail_dist,
    }

In [10]:
NUM_INSTANCES = 10
NUM_SCENARIOS = 500
rng = np.random.default_rng(2025)

max_start = T - NUM_SCENARIOS
if max_start <= 0:
    raise ValueError(f"Need at least {NUM_SCENARIOS} time steps; have {T}")

# Random start indices for each instance (non-overlapping or random)
start_indices = rng.integers(0, max_start + 1, size=NUM_INSTANCES)

results_list = []
for i in range(NUM_INSTANCES):
    start = int(start_indices[i])
    probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
        solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
        solar_buses, wind_buses, start_idx=start, num_scenarios=NUM_SCENARIOS, rng=rng
    )
    print(f"the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are {probs.shape}, {mc_g_i.shape}, {mv_d_j.shape}, {g_i_bar.shape}, {d_j_bar.shape}")
    res = run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar, logfile=debug_log)
    results_list.append(res)
    print(f"Instance {i+1}/{NUM_INSTANCES} (start={start}) done.")

the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are (500,), (27,), (1,), (500, 27), (500, 1)


/opt/anaconda3/envs/csci2470/lib/python3.11/site-packages/cvxpy/problems/problem.py:167: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "


                                     CVXPY                                     
                                     v1.7.1                                    


(CVXPY) May 03 05:18:18 PM: Your problem has 14028 variables, 28501 constraints, and 0 parameters.
(CVXPY) May 03 05:18:22 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 03 05:18:22 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 03 05:18:22 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 03 05:18:22 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) May 03 05:18:24 PM: Compiling problem (target solver=GUROBI).
(CVXPY) May 03 05:18:24 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) May 03 05:18:24 PM: Applying reduction CvxAttr2Constr
(CVXPY) May 03 05:18:24 PM: Applying reduction Qp2SymbolicQp


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


...

-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) May 03 05:53:17 PM: Applying reduction Qp2SymbolicQp
(CVXPY) May 03 05:53:30 PM: Applying reduction QpMatrixStuffing
(CVXPY) May 03 05:55:36 PM: Applying reduction GUROBI
(CVXPY) May 03 05:55:36 PM: Finished problem compilation (took 1.499e+02 seconds).
(CVXPY) May 03 05:55:36 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G325)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Non-default parameters:
QCPDual  1

Optimize a model with 141001 rows, 70529 columns and 323028 nonzeros
Model fingerprint: 0xf683e14e
Coefficient statistics:
  Matrix range     [8e-01, 1e+03]
  Objective range  [1e-04, 8e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-02, 1e+05]
Presolve removed 84000 rows and 2889 columns
Presolve time: 0.10s
Presolved: 57001 rows, 67640 columns, 221694 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier lo

(CVXPY) May 03 05:55:42 PM: Problem status: optimal
(CVXPY) May 03 05:55:42 PM: Optimal value: -5.474e+07
(CVXPY) May 03 05:55:42 PM: Compilation took 1.499e+02 seconds
(CVXPY) May 03 05:55:42 PM: Solver (including time spent in interface) took 4.883e+00 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Instance 10/10 (start=8061) done.


In [11]:
# Aggregate and average
keys = list(results_list[0].keys())
means = {k: np.mean([r[k] for r in results_list]) for k in keys}
stds = {k: np.std([r[k] for r in results_list]) for k in keys}

print("============== Real-data results (averaged over {} instances) ================".format(NUM_INSTANCES))
print("Distortion (DA vs E[RT price]):")
print("  Stochastic:", means["stoch_distortion"], "±", stds["stoch_distortion"])
print("  CVaR:", means["cvar_distortion"], "±", stds["cvar_distortion"])
print("  Deterministic:", means["det_distortion"], "±", stds["det_distortion"])
print("E[Social Surplus]:")
print("  Stochastic:", means["stoch_ss"], "±", stds["stoch_ss"])
print("  CVaR:", means["cvar_ss"], "±", stds["cvar_ss"])
print("  Deterministic:", means["det_ss"], "±", stds["det_ss"])
print("Tail (5%) welfare (mean positive SS in worst tail):")
print("  Stochastic:", means["stoch_tail_welfare"], "±", stds["stoch_tail_welfare"])
print("  CVaR:", means["cvar_tail_welfare"], "±", stds["cvar_tail_welfare"])
print("  Deterministic:", means["det_tail_welfare"], "±", stds["det_tail_welfare"])
print("Tail (5%) price distortion:")
print("  Stochastic:", means["stoch_tail_distortion"], "±", stds["stoch_tail_distortion"])
print("  CVaR:", means["cvar_tail_distortion"], "±", stds["cvar_tail_distortion"])
print("  Deterministic:", means["det_tail_distortion"], "±", stds["det_tail_distortion"])
print("Probability feasible:", means["prob_feasible"], "±", stds["prob_feasible"])

============== Real-data results (averaged over 10 instances) ================
Distortion (DA vs E[RT price]):
  Stochastic: 0.2417335294489135 ± 0.07545903738203191
  CVaR: 110.31372964009522 ± 0.06349479263490834
  Deterministic: 403.84139765569336 ± 18.56224819337082
E[Social Surplus]:
  Stochastic: 50439960.04080574 ± 482445.81605449645
  CVaR: 50438313.99019052 ± 482662.2239405528
  Deterministic: 49708871.90922298 ± 462336.8313540358
Tail (5%) welfare (mean positive SS in worst tail):
  Stochastic: 47831771.58097801 ± 181299.2923707616
  CVaR: 47865233.65880746 ± 181328.01298853
  Deterministic: 47801680.89850707 ± 178538.57796445588
Tail (5%) price distortion:
  Stochastic: 100.00000000000027 ± 8.597625825036818e-13
  CVaR: 9.146114949937464e-13 ± 7.117282853320532e-13
  Deterministic: 100.0 ± 0.0
Probability feasible: 0.020450221882226713 ± 0.010027050792888847


In [12]:
summary = pd.DataFrame({
    "Method": ["Stochastic", "CVaR", "Deterministic"] * 3,
    "Metric": ["Distortion", "Distortion", "Distortion", "E[SS]", "E[SS]", "E[SS]", "Tail welfare", "Tail welfare", "Tail welfare"],
    "Mean": [
        means["stoch_distortion"], means["cvar_distortion"], means["det_distortion"],
        means["stoch_ss"], means["cvar_ss"], means["det_ss"],
        means["stoch_tail_welfare"], means["cvar_tail_welfare"], means["det_tail_welfare"],
    ],
    "Std": [
        stds["stoch_distortion"], stds["cvar_distortion"], stds["det_distortion"],
        stds["stoch_ss"], stds["cvar_ss"], stds["det_ss"],
        stds["stoch_tail_welfare"], stds["cvar_tail_welfare"], stds["det_tail_welfare"],
    ],
})
summary["Std/Mean"] = summary["Std"] / summary["Mean"]
display(summary)

,Method,Metric,Mean,Std,Std/Mean
0,Stochastic,Distortion,2.417335e-01,0.075459,0.312158
1,CVaR,Distortion,1.103137e+02,0.063495,0.000576
2,Deterministic,Distortion,4.038414e+02,18.562248,0.045964
3,Stochastic,E[SS],5.043996e+07,482445.816054,0.009565
4,CVaR,E[SS],5.043831e+07,482662.223941,0.009569
5,Deterministic,E[SS],4.970887e+07,462336.831354,0.009301
6,Stochastic,Tail welfare,4.783177e+07,181299.292371,0.003790
7,CVaR,Tail welfare,4.786523e+07,181328.012989,0.003788
8,Deterministic,Tail welfare,4.780168e+07,178538.577964,0.003735


In [13]:
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("\n=== Summary dataframe ===")
print(summary.to_string(index=False))

with open(LOG_DIR / "summary_dataframe.log", "w") as f:
    f.write(summary.to_string(index=False))
    f.write("\n")


=== Summary dataframe ===
       Method       Metric         Mean           Std  Std/Mean
   Stochastic   Distortion 2.417335e-01      0.075459  0.312158
         CVaR   Distortion 1.103137e+02      0.063495  0.000576
Deterministic   Distortion 4.038414e+02     18.562248  0.045964
   Stochastic        E[SS] 5.043996e+07 482445.816054  0.009565
         CVaR        E[SS] 5.043831e+07 482662.223941  0.009569
Deterministic        E[SS] 4.970887e+07 462336.831354  0.009301
   Stochastic Tail welfare 4.783177e+07 181299.292371  0.003790
         CVaR Tail welfare 4.786523e+07 181328.012989  0.003788
Deterministic Tail welfare 4.780168e+07 178538.577964  0.003735
